# Practice 3: Customer Clustering - Bước 5: Xử lý đặc trưng (Feature Engineering)

---

## 1. Import các thư viện và tải dữ liệu đã làm sạch

Tải tệp `Train_cleaned.csv` từ bước làm sạch dữ liệu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/Train_cleaned.csv')
print(f'Kích thước dữ liệu sạch: {df.shape}')

## 2. Phân tách tập dữ liệu

Chúng ta tách cột định danh `ID` và cột nhãn phân khúc `Segmentation` (nhãn thực tế của doanh nghiệp) ra khỏi tập đặc trưng huấn luyện $X$.

In [ ]:
customer_ids = df['ID'].values
y_true = df['Segmentation'].values # Nhãn thực tế dùng để đối chiếu ở bước sau

X = df.drop(columns=['ID', 'Segmentation'])
print('Các cột đặc trưng đầu vào X:')
print(list(X.columns))

## 3. Áp dụng Log Transformation cho biến lệch `Work_Experience` từ Scratch

Như thảo luận, `Work_Experience` lệch phải rất mạnh và chứa giá trị 0. Ta áp dụng phép biến đổi $f(x) = \log(x + 1)$ để làm mượt phân phối trước khi chuẩn hóa thang đo.

In [ ]:
# Tạo hàm log1p từ scratch
def custom_log1p(X_col):
    return np.log(np.array(X_col) + 1.0)

# Áp dụng cho Work_Experience (Lấy trực tiếp từ df thô để tránh lỗi khi chạy lại ô code nhiều lần)
X['Work_Experience'] = custom_log1p(df['Work_Experience'])
print('Mẫu giá trị Work_Experience sau phép biến đổi log(x+1):')
print(X['Work_Experience'].head())

# Trực quan hóa so sánh phân phối trước và sau biến đổi Log(x+1)
plt.figure(figsize=(12, 5))

# Phân phối thô ban đầu
plt.subplot(1, 2, 1)
sns.histplot(df['Work_Experience'], kde=True, color='red', bins=15)
plt.title('Phân phối Work_Experience gốc (Lệch phải)')
plt.xlabel('Số năm kinh nghiệm')
plt.ylabel('Tần suất')

# Phân phối sau phép biến đổi log(x+1)
plt.subplot(1, 2, 2)
sns.histplot(X['Work_Experience'], kde=True, color='green', bins=15)
plt.title('Phân phối Work_Experience sau Log(x+1)')
plt.xlabel('log(Work_Experience + 1)')
plt.ylabel('Tần suất')

plt.tight_layout()
plt.show()

## 4. Tự viết bộ chuẩn hóa dữ liệu số học (Custom Scalers từ Scratch)

Chúng ta sẽ xây dựng lớp `CustomMinMaxScaler` từ scratch để đưa tất cả các đặc trưng số về khoảng $[0, 1]$, cân bằng thang đo với các biến One-Hot.

In [ ]:
class CustomMinMaxScaler:
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.min_ = None
        self.max_ = None
        self.range_ = None
        
    def fit(self, X):
        X_arr = np.array(X)
        self.min_ = np.min(X_arr, axis=0)
        self.max_ = np.max(X_arr, axis=0)
        self.range_ = self.max_ - self.min_
        # Tránh lỗi chia cho 0 nếu min == max
        self.range_ = np.where(self.range_ == 0, 1e-8, self.range_)
        return self
        
    def transform(self, X):
        X_arr = np.array(X)
        X_std = (X_arr - self.min_) / self.range_
        return X_std * (self.feature_range[1] - self.feature_range[0]) + self.feature_range[0]
        
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

## 5. Tự viết bộ mã hóa biến phân loại từ Scratch

Mã hóa `Spending_Score` dạng Ordinal và One-Hot cho các nominal features còn lại.

In [ ]:
# 5.1. Ordinal Encoding cho Spending_Score
spending_mapping = {'Low': 0, 'Average': 1, 'High': 2}
X['Spending_Score'] = X['Spending_Score'].map(spending_mapping)

# 5.2. Custom One-Hot Encoder
class CustomOneHotEncoder:
    def __init__(self):
        self.categories_ = {}
        
    def fit(self, df, columns):
        self.columns = columns
        for col in columns:
            self.categories_[col] = sorted(list(df[col].unique()))
        return self
        
    def transform(self, df):
        df_out = df.copy()
        for col in self.columns:
            cats = self.categories_[col]
            for cat in cats:
                new_col_name = f'{col}_{cat}'
                df_out[new_col_name] = (df_out[col] == cat).astype(int)
            df_out = df_out.drop(columns=[col])
        return df_out
        
    def fit_transform(self, df, columns):
        self.fit(df, columns)
        return self.transform(df)

nominal_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Var_1']
encoder = CustomOneHotEncoder()
X_encoded = encoder.fit_transform(X, nominal_cols)
print(f'Hình dạng ma trận sau One-Hot: {X_encoded.shape}')

## 6. Tự viết lớp Giảm chiều dữ liệu PCA từ Scratch (Custom PCA)

Chúng ta tự xây dựng thuật toán PCA dựa trên đại số tuyến tính để khảo sát mức độ cô đọng thông tin của dữ liệu thưa.

In [ ]:
class CustomPCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.eigenvalues_ = None
        
    def fit(self, X):
        X_arr = np.array(X, dtype=float)
        # 1. Định tâm dữ liệu
        self.mean_ = np.mean(X_arr, axis=0)
        X_centered = X_arr - self.mean_
        
        # 2. Tính ma trận hiệp phương sai
        cov_matrix = np.cov(X_centered, rowvar=False)
        
        # 3. Tìm trị riêng và vectơ riêng
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # 4. Sắp xếp giảm dần
        sorted_indices = np.argsort(eigenvalues)[::-1]
        self.eigenvalues_ = eigenvalues[sorted_indices]
        sorted_eigenvectors = eigenvectors[:, sorted_indices]
        
        # Chọn n_components đầu tiên
        self.components_ = sorted_eigenvectors[:, :self.n_components]
        return self
        
    def transform(self, X):
        X_arr = np.array(X, dtype=float)
        X_centered = X_arr - self.mean_
        return np.dot(X_centered, self.components_)
        
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
        
    def explained_variance_ratio(self):
        total_var = np.sum(self.eigenvalues_)
        return self.eigenvalues_[:self.n_components] / total_var

## 7. Xây dựng quy trình xử lý đặc trưng và Khảo sát PCA

Chúng ta thực hiện chuẩn hóa khoảng $[0, 1]$ cho các cột số để cân bằng tuyệt đối với các thuộc tính One-Hot (MinMax Scaling).

In [ ]:
# Chuẩn hóa MinMax cho các biến số học
numerical_cols = ['Age', 'Work_Experience', 'Family_Size']
scaler_minmax = CustomMinMaxScaler()

X_scaled = X_encoded.copy()
X_scaled[numerical_cols] = scaler_minmax.fit_transform(X_encoded[numerical_cols])

print('Mẫu dữ liệu sau khi chuẩn hóa MinMax [0, 1]:')
print(X_scaled[numerical_cols].head(3))

In [ ]:
# Khảo sát giảm chiều PCA để xem lượng thông tin giữ lại
pca = CustomPCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print(f'Tỷ lệ phương sai giải thích được của 3 thành phần: {pca.explained_variance_ratio()}')
print(f'Tổng lượng thông tin giữ lại ở không gian 3D: {np.sum(pca.explained_variance_ratio())*100:.2f}%')
print('\n-> Nhận xét: Lượng thông tin giải thích được của 3 thành phần chính quá thấp (55.34%).')
print('Do đó, ta quyết định KHÔNG dùng tập giảm chiều PCA để huấn luyện mô hình.')
print('Toàn bộ các bước huấn luyện phía sau sẽ sử dụng bộ dữ liệu đầy đủ 22 chiều để giữ lại 100% thông tin.')

## 8. Tự viết hàm chia tập dữ liệu từ Scratch

Chia tách dữ liệu thành tập Train (80%) và Validation (20%) sử dụng hàm tự chế.

In [ ]:
def train_val_split_scratch(X_data, y_data, val_size=0.2, random_state=42):
    np.random.seed(random_state)
    n_samples = len(X_data)
    n_val = int(n_samples * val_size)
    
    shuffled_indices = np.random.permutation(n_samples)
    val_indices = shuffled_indices[:n_val]
    train_indices = shuffled_indices[n_val:]
    
    if isinstance(X_data, pd.DataFrame):
        X_train = X_data.iloc[train_indices]
        X_val = X_data.iloc[val_indices]
    else:
        X_train = X_data[train_indices]
        X_val = X_data[val_indices]
        
    y_train = y_data[train_indices]
    y_val = y_data[val_indices]
    
    return X_train, X_val, y_train, y_val

# Chia tách cho bộ dữ liệu đầy đủ chiều (scaled)
X_train, X_val, y_train, y_val = train_val_split_scratch(X_scaled, y_true, val_size=0.2, random_state=42)

print(f'Kích thước dữ liệu huấn luyện Train: {X_train.shape}')
print(f'Kích thước dữ liệu xác thực Val:    {X_val.shape}')

## 9. Trực quan hóa dữ liệu đặc trưng huấn luyện trước khi lưu trữ

Chúng ta in ra 20 dòng dữ liệu đặc trưng huấn luyện đầu tiên (`X_train.head(20)`) để trực quan hóa cấu trúc dữ liệu sau khi kết thúc xử lý đặc trưng.

In [ ]:
print('20 dòng dữ liệu đặc trưng huấn luyện đầu tiên (X_train):')
display(X_train.head(20))

## 10. Lưu trữ dữ liệu xử lý đặc trưng đầy đủ

Chúng ta sẽ xuất bộ đặc trưng đầy đủ 22 chiều và nhãn tương ứng để làm đầu vào cho các thuật toán phân cụm.

In [ ]:
os.makedirs('../data/processed_features/', exist_ok=True)

# Lưu tập Full scaled
X_train.to_csv('../data/processed_features/X_train.csv', index=False)
X_val.to_csv('../data/processed_features/X_val.csv', index=False)

# Lưu nhãn thực tế
pd.DataFrame(y_train, columns=['Segmentation']).to_csv('../data/processed_features/y_train.csv', index=False)
pd.DataFrame(y_val, columns=['Segmentation']).to_csv('../data/processed_features/y_val.csv', index=False)

print('Đã lưu toàn bộ dữ liệu đặc trưng đầy đủ 22 chiều thành công!')

## 11. Xác nhận từ học viên

Bạn hãy thực thi toàn bộ notebook này và xác nhận đã sẵn sàng chuyển sang bước tiếp theo.